In [6]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\universitetler_qebul_2021.csv")
df.head()

,universitet,tip,qebul_plani,orta_bal_I_IV,orta_bal_V
0,Bakı Dövlət Universiteti,Dövlət,6082,407.631,123.511
1,Azərbaycan Dövlət Neft və Sənaye Universiteti,Dövlət,3340,392.367,NaN
2,Azərbaycan Texniki Universiteti,Dövlət,2235,304.043,NaN
3,Azərbaycan Memarlıq və İnşaat Universiteti,Dövlət,1970,291.500,161.672
4,Azərbaycan Tibb Universiteti,Dövlət,845,596.884,NaN


Məlumatların yüklənməsi və ilkin baxış:
Bu blokda təhlil üçün tələb olunan Python kitabxanaları daxil edilir və 2021-ci il ali məktəblərə qəbul göstəricilərini əks etdirən CSV faylı bazaya yüklənir. Mənbə: Dövlət İmtahan Mərkəzi (DİM), "Elmi-statistik təhlil", 2021/2022-ci tədris ili, "Abituriyent" jurnalı №12, 2021.

In [9]:
print("TƏSVİRİ STATİSTİKA")
print("=" * 70)
print(f"Ümumi universitet sayı: {len(df)}")
print(df["tip"].value_counts().to_string(), end="\n\n")
print(df[["qebul_plani", "orta_bal_I_IV", "orta_bal_V"]].describe().round(2).to_string())

TƏSVİRİ STATİSTİKA
Ümumi universitet sayı: 42
tip
Dövlət    30
Özəl      12

       qebul_plani  orta_bal_I_IV  orta_bal_V
count        42.00          39.00       19.00
mean       1180.79         353.49      121.92
std        1197.80         100.01       30.03
min          72.00         205.91       84.66
25%         407.50         286.43      100.22
50%         927.50         330.36      116.09
75%        1622.50         389.54      133.42
max        6082.00         631.13      192.20


Təsviri Statistika Təhlili:
Təhlil olunan 42 universitetin çox hissəsini dövlət alimləri təşkil edir (30 dövlət, 12 özəl) və qəbul planında minimum 72, maksimum 6082 nəfər arasında kəskin fərq, yüksək standart meyl (std = 1197.80) var. Orta ballar üzrə I–IV ixtisas qrupları üzrə göstərici ortalama 353.49 bal təşkil etdiyi halda, V qrup üzrə bu göstərici 121.92 bal səviyyəsindədir.

In [15]:
print("MANN WHITNEY U: Dövlət vs Özəl universitetlərin orta balı (I-IV qrup)")
print("=" * 70)

state = df.loc[df["tip"] == "Dövlət", "orta_bal_I_IV"].dropna()
priv = df.loc[df["tip"] == "Özəl", "orta_bal_I_IV"].dropna()

print("state_p=", stats.shapiro(state).pvalue)
print("priv_p=", stats.shapiro(priv).pvalue)

u_stat, p_value = stats.mannwhitneyu(state, priv, alternative="two-sided")

print(f"Dövlət median = {state.median():.2f}")
print(f"Özəl median   = {priv.median():.2f}")
print(f"U = {u_stat:.1f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Nəticə: Statistik əhəmiyyətli fərq (p<0.05)")
else:
    print("Nəticə: Statistik əhəmiyyətli fərq yoxdur")
print()


MANN WHITNEY U: Dövlət vs Özəl universitetlərin orta balı (I-IV qrup)
state_p= 0.010061713877435825
priv_p= 0.7853980629304695
Dövlət median = 362.71
Özəl median   = 277.02
U = 273.0, p = 0.0008
Nəticə: Statistik əhəmiyyətli fərq (p<0.05)



Mann-Whitney U Testi Şərhi: Shapiro-Wilk testi dövlət universitetlərinin bal paylanmasının normal olmadığını gösterdiyi üçün  paramerik olmayan Mann-Whitney U testi tətbiq olunmuşdur. Analiz nəticəsində dövlət universitetlərinin median balının (362.71) özəl universitetlərdən (277.02) statistik olaraq əhəmiyyətli dərəcədə yüksək olduğu müəyyən edilmişdir.

In [16]:
print("PEARSON KORRELYASİYA: qəbul planı vs orta bal (I-IV qrup)")
print("=" * 70)

sub = df.dropna(subset=["qebul_plani", "orta_bal_I_IV"])
r, p_value = stats.pearsonr(sub["qebul_plani"], sub["orta_bal_I_IV"])

print(f"n = {len(sub)}")
print(f"r = {r:.3f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("Nəticə: Əhəmiyyətli xətti əlaqə (p<0.05)")
else:
    print("Nəticə: Xətti əlaqə statistik əhəmiyyətli deyil")

PEARSON KORRELYASİYA: qəbul planı vs orta bal (I-IV qrup)
n = 39
r = 0.017, p = 0.9187
Nəticə: Xətti əlaqə statistik əhəmiyyətli deyil


Pearson Korrelyasiya Təhlili: Qəbul planı ilə I–IV qrup üzrə orta bal arasındakı xətti əlaqəni qiymətləndirmək üçün 39 universitet üzrə Pearson korrelyasiya testi tətbiq olunmuşdur. Analiz nəticəsində dəyişənlər arasında sıfıra yaxın zəif əlaqə müşahidə edilmiş və bu münasibətin statistik olaraq əhəmiyyətli olmadığı müəyyən olunmuşdur.

In [17]:
print("OLS REQRESSİYA: orta_bal_I_IV ~ qebul_plani + tip_ozel")
print("=" * 70)

sub = sub.copy()
sub["tip_ozel"] = (sub["tip"] == "Özəl").astype(int)

X = sm.add_constant(sub[["qebul_plani", "tip_ozel"]])
y = sub["orta_bal_I_IV"]
model = sm.OLS(y, X).fit()

print(model.summary())

OLS REQRESSİYA: orta_bal_I_IV ~ qebul_plani + tip_ozel
                            OLS Regression Results                            
Dep. Variable:          orta_bal_I_IV   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.185
Method:                 Least Squares   F-statistic:                     5.318
Date:                Fri, 07 Aug 2026   Prob (F-statistic):            0.00947
Time:                        20:15:19   Log-Likelihood:                -229.39
No. Observations:                  39   AIC:                             464.8
Df Residuals:                      36   BIC:                             469.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------

OLS Reqressiya Analizi: Qurulan OLS reqressiya modeli ümumilikdə statistik əhəmiyyətlidir və orta bal dəyişkənliyinin 22.8%-ini izah edir. Model nəticələrinə əsasən, qəbul planının orta bala təsiri statistik əhəmiyyətsiz olduğu halda, universitetin özəl olması orta balı dövlət universitetlərinə nəzərən təxminən 106.19 bal aşağı salır və bu fərq statistik əhəmiyyətlidir.